In [27]:
import os
from pathlib import Path
from omegaconf import OmegaConf
from copy import deepcopy
from typing import List, Union, Dict, Any

import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, classification_report

os.chdir("../src")  # Change to project root directory

from config import DatasetConfig, ConceptDatasetConfig
from models.architectures import CBMSequentialEfficientNetFCN
from config import CBMTrainerConfig
from models.trainer.cbm_trainer import CBMTrainer
from models.architectures import EfficientNetv2
from models.trainer.standard_trainer import StandardTrainer
from config.training_config import TrainingConfig
from config.training_config import ConceptTrainingConfig

from rule_eval import construct_full_graph
from analysis_utils import (
    get_dataset_predictions,
    analyze_fuzzy_loss_single_model,
    compare_fuzzy_losses,
    analyze_rule_violations,
    compare_violations,
    print_fuzzy_loss_results,
    print_violation_results,
)

# Functions

In [28]:

def load_config(config_path, configClass = CBMTrainerConfig, overrides: Union[List[str], Dict[str, Any], None] = None):
    """
    Load the configuration from a YAML file.
    """
    # Load YAML
    cfg_yaml = OmegaConf.load(config_path)
    cfg_structured = OmegaConf.structured(configClass)
    cfg = OmegaConf.merge(cfg_structured, cfg_yaml)
    cfg = OmegaConf.to_object(cfg)

    if overrides is not None:
        overrides_cfg = None
        if isinstance(overrides, list):
            # Create a DictConfig from a dot-list (e.g., ["model.hidden_size=512", "lr=0.001"])
            overrides_cfg = OmegaConf.from_dotlist(overrides)
        elif isinstance(overrides, dict):
            # Create a DictConfig from a standard dictionary
            overrides_cfg = OmegaConf.create(overrides)
        
        if overrides_cfg is not None:
            cfg = OmegaConf.merge(cfg, overrides_cfg)
            cfg = OmegaConf.to_object(cfg)

    cfg.resolve()
    return cfg

# Loading configs

In [29]:
concept_overrides = [
        "device_no=0",
        "dataset.name=gtsrb",
        "dataset.n_labels=43",
        "dataset.data_path=../data/raw/GTSRB/converted",
        "dataset.n_concepts=43",
        "dataset.concepts_file=../data/raw/GTSRB/concepts/concepts_per_class.csv"
        ]

fuzzy_config = load_config(
    Path("../files/configs/GTSRB_CBM_config_best_trial_loading.yaml"),
    overrides=concept_overrides
    )

baseline_config = load_config(
    Path("../files/configs/GTSRB_CBM_config_loading.yaml"),
    overrides=concept_overrides
    )

Directory 'experiments/20251022_183858' created successfully.
Directory 'experiments/20251022_183858' created successfully.


# Load Dataset

In [30]:
# Concept dataset
(concept_train_loader,
 concept_val_loader,
 concept_test_loader) = (baseline_config.dataset
                            .factory(
                                seed=baseline_config.seed, 
                                config=baseline_config.dataset
                            )
                            .get_dataloaders()
                        )

# Load trained Models

In [31]:
import torch
from pathlib import Path
fuzzy_cbm_label_precitor_path = Path("../experiments/fuzzy_CBM/models/20251001_113637_label_predictor_best_model.pt")
def load_cbm_model(config, directory):
    model=CBMSequentialEfficientNetFCN(config)
    model.concept_predictor.load_state_dict(
    torch.load(directory, map_location=config.device, weights_only=True)
    )
    model.label_predictor.load_state_dict(
        torch.load(fuzzy_cbm_label_precitor_path, map_location=config.device, weights_only=True)
    )
    return model

In [32]:
rq1_results_path = Path("../notebooks/")

In [33]:
rule_checker = construct_full_graph()

In [34]:
fuzzy_config.device

'cuda:0'

In [35]:
from analysis_utils import get_dataset_predictions
 
 
 
 
 
def get_violation_count(models_path, model_config, concept_pred_threshold, tag, dataloader):
    model_names = os.listdir(models_path)
    concepts_df = pd.DataFrame()
 
    i = 0
 
    for model_name in model_names:
        print(f"iteration {i}")
        print(f"Evaluating Baseline CBM model: {model_name}")
        seed=model_name.split("_")[2]
        model_directory = models_path / model_name
        model = load_cbm_model(model_config, model_directory)
        model.to(model_config.device)
       
        preds_gtsrb = get_dataset_predictions(
            model, dataloader, model_config.device, f"GTSRB ({tag})"
        )
       
        rule_viols_gtsrb = analyze_rule_violations(
            preds_gtsrb['predictions'],
            'GTSRB',
            tag,
            rule_checker
        )
 
        concepts_df = pd.concat([
                    concepts_df,
                    pd.DataFrame([{"seed":seed, "violations":rule_viols_gtsrb['total_violations']}])
                ]
            )
       
 
       
        i += 1
    return concepts_df

In [36]:
models_path = rq1_results_path / "fuzzy_cbm_30" 
model_config = fuzzy_config
concept_pred_threshold = 0.5
tag = "ReqAware"
fuzzy_violation_count = get_violation_count(models_path, model_config, concept_pred_threshold, tag, concept_test_loader)

iteration 0
Evaluating Baseline CBM model: 20251010_102822_s42_concept_predictor_best_model.pt


Getting GTSRB (ReqAware) predictions: 100%|██████████| 99/99 [00:14<00:00,  6.79it/s]


iteration 1
Evaluating Baseline CBM model: 20251020_223800_s469_concept_predictor_best_model.pt


Getting GTSRB (ReqAware) predictions: 100%|██████████| 99/99 [00:13<00:00,  7.45it/s]


iteration 2
Evaluating Baseline CBM model: 20251020_223800_s263_concept_predictor_best_model.pt


...

Getting GTSRB (ReqAware) predictions: 100%|██████████| 99/99 [00:13<00:00,  7.42it/s]


iteration 29
Evaluating Baseline CBM model: 20251020_223819_s64_concept_predictor_best_model.pt


Getting GTSRB (ReqAware) predictions: 100%|██████████| 99/99 [00:13<00:00,  7.38it/s]


iteration 30
Evaluating Baseline CBM model: 20251020_223819_s714_concept_predictor_best_model.pt


Getting GTSRB (ReqAware) predictions: 100%|██████████| 99/99 [00:13<00:00,  7.25it/s]


In [ ]:
fuzzy_violation_count.plot(kind="box")

In [ ]:
fuzzy_violation_count.sort_values(by="violations")

In [ ]:
models_path = rq1_results_path / "baseline_cbm_30" 
model_config = fuzzy_config
concept_pred_threshold = 0.5
tag = "ReqAware"
baseline_violation_count = get_violation_count(models_path, model_config, concept_pred_threshold, tag, concept_test_loader)

iteration 0
Evaluating Baseline CBM model: 20251016_224526_s263_baseline_concept_predictor_best_model.pt


Getting GTSRB (ReqAware) predictions: 100%|██████████| 99/99 [00:13<00:00,  7.49it/s]


iteration 1
Evaluating Baseline CBM model: 20251016_224526_s469_baseline_concept_predictor_best_model.pt


Getting GTSRB (ReqAware) predictions:  30%|███       | 30/99 [00:05<00:08,  8.27it/s]